# Database indexes
Index columns used often for filtering and ordering.


In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE tasks (id INTEGER PRIMARY KEY, owner_id INTEGER, created_at TEXT)")
db.execute("CREATE INDEX idx_tasks_owner ON tasks(owner_id)")
plan = db.execute("EXPLAIN QUERY PLAN SELECT * FROM tasks WHERE owner_id = ?", (7,)).fetchall()
print(plan)


## Polished version
Match a composite index to the query's filter and sort order, then inspect the plan.


In [ ]:
class IndexedTaskRepository:
    def __init__(self, connection: sqlite3.Connection) -> None:
        self.connection = connection
        self.connection.execute(
            "CREATE INDEX IF NOT EXISTS idx_tasks_owner_created ON tasks(owner_id, created_at DESC)"
        )

    def recent_for_owner(self, owner_id: int) -> list[tuple]:
        return self.connection.execute(
            "SELECT id, owner_id, created_at FROM tasks WHERE owner_id = ? ORDER BY created_at DESC",
            (owner_id,),
        ).fetchall()

repository = IndexedTaskRepository(db)
query_plan = db.execute(
    "EXPLAIN QUERY PLAN SELECT * FROM tasks WHERE owner_id = ? ORDER BY created_at DESC",
    (7,),
).fetchall()
print(query_plan)
